In [ ]:
#%%appyter init
from appyter import magic
magic.init(lambda _=globals: _())


# Drug & Gene Perturbation Discovery Appyter
**Perturb-Seqr → DrugEnrichr → Enrichr**

This appyter takes a gene set — either a single list, or an up-regulated/down-regulated pair — and chains together three tools to build a drug and gene perturbation hypothesis:

1. **Perturb-Seqr** searches a large connectivity-mapping database of small-molecule and gene-knockout perturbation signatures for hits that mimic or reverse your gene signature.
2. **DrugEnrichr** takes the top drug hits from Perturb-Seqr and runs enrichment analysis to find their shared mechanisms of action, targets, and other drug-set annotations.
3. **Enrichr** takes the top gene-knockout hits from Perturb-Seqr and runs enrichment analysis to find enriched pathways and ontologies.

Fill in your gene set and the pipeline parameters below, then submit to run the analysis. Each step includes both a results table and a chart summarizing the enrichment/perturbation results. A link to the full, interactive results on each tool's website is provided at the bottom of its section.

> Tools used: [Perturb-Seqr](https://perturbseqr.maayanlab.cloud/), [DrugEnrichr](https://maayanlab.cloud/DrugEnrichr/), [Enrichr](https://maayanlab.cloud/Enrichr/)


In [ ]:
import re
import json
import difflib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display, HTML, Markdown

pd.set_option('display.max_colwidth', None)

# Error handling
class NoResults(Exception):
    pass
class APIFailure(Exception):
    pass


In [ ]:
%%appyter code_exec

{% do SectionField(
    name='INPUT',
    title='1. Gene Set Input',
    subtitle='Provide a single gene set, or an up-regulated / down-regulated gene pair.',
) %}

mode = "{{ ChoiceField(
    name='mode',
    label='Search mode',
    description='Single gene set enrichment, or a paired up/down-regulated gene set search (finds perturbations that mimic or reverse the signature)',
    choices={
        'Single gene set': 'single',
        'Paired up/down gene set': 'up_down',
    },
    default='Single gene set',
    section='INPUT',
) }}"

single_gene_set = {{ TextField(
    name='single_gene_set',
    label='Single gene set',
    description='Comma- or newline-separated gene symbols. Used when Search mode = "Single gene set".',
    default='TP53, MDM2, CDKN1A, BAX, BBC3, GADD45A, RB1, ATM, CHEK2, MDM4',
    section='INPUT',
) }}

genes_up_input = {{ TextField(
    name='genes_up_input',
    label='Up-regulated genes',
    description='Comma- or newline-separated gene symbols. Used when Search mode = "Paired up/down gene set".',
    default='IL6, TNF, IL1B, CXCL8, NFKB1, PTGS2',
    section='INPUT',
) }}

genes_down_input = {{ TextField(
    name='genes_down_input',
    label='Down-regulated genes',
    description='Comma- or newline-separated gene symbols. Used when Search mode = "Paired up/down gene set".',
    default='NR3C1, FOXP3, IL10, TGFB1',
    section='INPUT',
) }}

{% do SectionField(
    name='PARAMS',
    title='2. Pipeline Parameters',
    subtitle='Control how many top hits flow through each step, and which libraries to enrich against.',
) %}

query_name = {{ StringField(
    name='query_name',
    label='Query name',
    description='A short label used to title tables/charts and name the downloaded report',
    default='my_query',
    section='PARAMS',
) }}

top_n_drugs = {{ IntField(
    name='top_n_drugs',
    label='Top N drugs',
    description='Number of top drug hits to carry forward from Perturb-Seqr into DrugEnrichr',
    min=1, max=100,
    default=20,
    section='PARAMS',
) }}

top_n_ko_genes = {{ IntField(
    name='top_n_ko_genes',
    label='Top N KO genes',
    description='Number of top gene-knockout hits to carry forward from Perturb-Seqr into Enrichr',
    min=1, max=100,
    default=20,
    section='PARAMS',
) }}

rank_by = "{{ ChoiceField(
    name='rank_by',
    label='Rank Perturb-Seqr hits by',
    description='Aggregated "consensus" score (robust across repeated signatures), or individual per-signature "enrichment" hits (matches the website\'s default view)',
    choices={
        'Per-signature enrichment (site default)': 'enrichment',
        'Aggregated consensus score': 'consensus',
    },
    default='Per-signature enrichment (site default)',
    section='PARAMS',
) }}"

drugenrichr_library = {{ StringField(
    name='drugenrichr_library',
    label='DrugEnrichr library',
    description='Drug-set library to enrich the top drugs against',
    default='SIDER_Side_Effects',
    section='PARAMS',
) }}

enrichr_library = {{ StringField(
    name='enrichr_library',
    label='Enrichr library',
    description='Gene-set library to enrich the top KO genes against',
    default='KEGG_2021_Human',
    section='PARAMS',
) }}


In [ ]:
def parse_genes(s):
    return [g.strip().upper() for g in s.replace("\n", ",").split(",") if g.strip()]

if mode == "single":
    input_genes = parse_genes(single_gene_set)
    genes_up, genes_down = [], []
    print(f"Single gene set loaded: {len(input_genes)} genes")
    print(input_genes)
else:
    genes_up = parse_genes(genes_up_input)
    genes_down = parse_genes(genes_down_input)
    input_genes = list(set(genes_up) | set(genes_down))
    print(f"Up genes loaded ({len(genes_up)}):")
    print(genes_up)
    print(f"Down genes loaded ({len(genes_down)}):")
    print(genes_down)


## Shared helper: bar chart
Used to visualize each step's top results.

In [ ]:
def make_bar_chart(labels, scores, title, xlabel, colors='lightskyblue', legend_handles=None, figsize=None):
    labels = list(labels)
    scores = list(scores)
    if figsize is None:
        figsize = (9, max(3, 0.4 * len(labels)))
    order = np.argsort(scores)  # ascending so the highest score plots at the top
    labels_sorted = [labels[i] for i in order]
    scores_sorted = [scores[i] for i in order]
    if isinstance(colors, (list, tuple, np.ndarray)):
        colors_sorted = [colors[i] for i in order]
    else:
        colors_sorted = colors

    plt.figure(figsize=figsize)
    ax = plt.gca()
    ax.barh(labels_sorted, scores_sorted, color=colors_sorted, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=16)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    if legend_handles:
        ax.legend(handles=legend_handles, loc='lower right', frameon=False)
    plt.tight_layout()
    return ax


## Step 1: Perturb-Seqr — Drug & Gene Perturbation Discovery

Queries the Perturb-Seqr GraphQL API. Depending on the mode chosen above, this runs either:

- **Single gene set enrichment** — finds any perturbation (drug or gene KO) signature that significantly overlaps your gene set
- **Paired up/down enrichment** — finds perturbations that *mimic* or *reverse* your up/down signature

Each perturbation hit returned by Perturb-Seqr can be either a **drug/small molecule** or a **gene knockout (KO)** —
the two are mixed together in a single ranked list. Perturb-Seqr can tell us which of its 18 datasets ("libraries")
each hit came from, and each library is exclusively either a drug dataset or a gene-perturbation dataset — so the
library a hit came from is a ground-truth signal for classifying it, far more reliable than trying to filter by
library *name* upfront. For any hit whose library name doesn't clearly match a known drug/gene keyword, this
appyter falls back to checking whether the hit's name resolves to a real gene symbol in Perturb-Seqr's own gene
database.

Results are ranked by significance, and — importantly — **not deduplicated**: the same drug or gene can
legitimately appear multiple times (e.g. `TP53` from several different cell lines/datasets), and every one of
those rows is kept exactly as Perturb-Seqr returned it.

In [ ]:
PERTURBSEQR_URL = "https://perturbseqr.maayanlab.cloud/graphql"

def _check_gql_errors(res):
    if "errors" in res and res["errors"]:
        raise APIFailure

def parse_perturbation_term(term):
    """Perturb-Seqr gene-set "term" strings pack the perturbation name plus a lot
    of metadata (cell line, tissue, dose, KO/KD/Mut label, dataset ID, etc.) separated
    by "::", e.g. "glucagon::::::::::human::GSE171352::2::3 down" or
    "TP53::CAL51::::::::KO::::SRP287851::2996 down". The perturbation name is always
    the FIRST "::"-delimited field -- everything else is metadata, not part of the name.
    The direction (up/down), when present, is the last whitespace-separated token."""
    fields = term.split("::")
    perturbation = fields[0].strip()
    last_field = fields[-1].strip() if fields else ""
    tokens = last_field.split(" ")
    direction = tokens[-1].lower() if len(tokens) > 1 and tokens[-1].lower() in ("up", "down") else None
    return perturbation, direction

def get_perturbation_name(term):
    """Convenience wrapper: just the perturbation name from a term string."""
    return parse_perturbation_term(term)[0]

def get_direction(term):
    """Convenience wrapper: just the direction ("up"/"down"/None) from a term string."""
    return parse_perturbation_term(term)[1]

def find_geneset_id_by_direction(nodes, direction):
    """Given a list of geneSet nodes (each a dict with a 'term' and an 'id'),
    return the id of the first node whose parsed direction matches `direction`
    ("up" or "down"), or None if none match. Uses an exact match on the parsed
    direction rather than a substring search, since a cell line or metadata
    field could otherwise contain "up"/"down" as part of another word."""
    return next(
        (node["id"] for node in nodes if get_direction(node["term"]) == direction),
        None,
    )

def enrich_perturbseqr_single_set(geneset, first=2000, library_names=None, filter_ko=False):
    variables = {
        "filterTerm": "",
        "offset": 0,
        "first": first,
        "filterFda": False,
        "sortBy": "pvalue_up",
        "filterKo": filter_ko,
        "genes": geneset,
    }
    if library_names is not None:
        variables["libraryNames"] = library_names

    query = {
        "operationName": "EnrichmentQuery",
        "variables": variables,
        "query": """query EnrichmentQuery(
                        $genes: [String]!
                        $filterTerm: String = ""
                        $offset: Int = 0
                        $first: Int = 10
                        $filterFda: Boolean = false
                        $sortBy: String = ""
                        $filterKo: Boolean = false
                        $libraryNames: [String]
                        ) {
                        currentBackground {
                            enrich(
                            genes: $genes
                            filterTerm: $filterTerm
                            offset: $offset
                            first: $first
                            filterFda: $filterFda
                            sortby: $sortBy
                            filterKo: $filterKo
                            libraryNames: $libraryNames
                            ) {
                            nodes {
                                geneSetHash
                                pvalue
                                adjPvalue
                                oddsRatio
                                nOverlap
                                geneSets {
                                nodes {
                                    term
                                    id
                                    nGeneIds
                                    geneSetFdaCountsById {
                                    nodes {
                                        approved
                                        count
                                    }
                                    }
                                    library {
                                    name
                                    }
                                }
                                totalCount
                                }
                            }
                            totalCount
                            geneSetCount
                            consensusCount
                            consensus {
                                drug
                                oddsRatio
                                pvalue
                                adjPvalue
                                approved
                                countSignificant
                                countInsignificant
                                countUpSignificant
                                pvalueUp
                                adjPvalueUp
                                oddsRatioUp
                                pvalueDown
                                adjPvalueDown
                                oddsRatioDown
                                libraries
                            }
                            }
                        }
                        }
                        """,
    }

    response = requests.post(PERTURBSEQR_URL, json=query)
    if not response.ok:
        raise APIFailure
    res = response.json()
    _check_gql_errors(res)
    consensus = res["data"]["currentBackground"]["enrich"]["consensus"]
    enrichment = res["data"]["currentBackground"]["enrich"]["nodes"]
    df_consensus = pd.DataFrame(consensus).rename(columns={"drug": "perturbation"})

    df_enrichment = pd.json_normalize(
        enrichment,
        record_path=["geneSets", "nodes"],
        meta=["geneSetHash", "pvalue", "adjPvalue", "oddsRatio", "nOverlap"],
    )
    if df_enrichment.empty:
        return pd.DataFrame(), df_consensus
    df_enrichment["approved"] = df_enrichment["geneSetFdaCountsById.nodes"].map(
        lambda x: x[0]["approved"] if len(x) > 0 else False
    )
    df_enrichment["count"] = df_enrichment["geneSetFdaCountsById.nodes"].map(
        lambda x: x[0]["count"] if len(x) > 0 else 0
    )
    df_enrichment.drop(columns=["geneSetFdaCountsById.nodes"], inplace=True)
    df_enrichment["direction"] = df_enrichment["term"].map(get_direction)
    df_enrichment["perturbation"] = df_enrichment["term"].map(get_perturbation_name)
    if "library.name" in df_enrichment.columns:
        df_enrichment = df_enrichment.rename(columns={"library.name": "library"})

    return df_enrichment, df_consensus


def enrich_perturbseqr_up_down(genes_up, genes_down, first=500, top_n=5000, filter_ko=False):
    query = {
        "operationName": "PairEnrichmentQuery",
        "variables": {
            "filterTerm": "",
            "offset": 0,
            "first": first,
            "filterFda": False,
            "sortBy": "pvalue_mimic",
            "filterKo": filter_ko,
            "topN": top_n,
            "pvalueLe": 0.05,
            "genesUp": genes_up,
            "genesDown": genes_down,
        },
        "query": """query PairEnrichmentQuery($genesUp: [String]!, $genesDown: [String]!, $filterTerm: String = "", $offset: Int = 0, $first: Int = 10, $filterFda: Boolean = false, $sortBy: String = "", $filterKo: Boolean = false, $topN: Int = 10000, $pvalueLe: Float = 0.05) {
          currentBackground {
            pairedEnrich(
              filterTerm: $filterTerm
              offset: $offset
              first: $first
              filterFda: $filterFda
              sortby: $sortBy
              filterKo: $filterKo
              topN: $topN
              pvalueLe: $pvalueLe
              genesDown: $genesDown
              genesUp: $genesUp
              ) {
                totalCount
                consensusCount
                consensus {
                  drug
                  oddsRatio
                  pvalue
                  adjPvalue
                  approved
                  countSignificant
                  countInsignificant
                  countUpSignificant
                  pvalueUp
                  adjPvalueUp
                  oddsRatioUp
                  pvalueDown
                  adjPvalueDown
                  oddsRatioDown
                  libraries
                  }
                  nodes {
                    adjPvalueMimic
                    adjPvalueReverse
                    mimickerOverlap
                    oddsRatioMimic
                    oddsRatioReverse
                    pvalueMimic
                    pvalueReverse
                    reverserOverlap
                    geneSet {
                      nodes {
                        id
                        nGeneIds
                        term
                        geneSetFdaCountsById {
                          nodes {
                            count
                            approved
                            }
                          }
                        library {
                          name
                          }
                        }
                      }
                    }
                  }
                }
              }
        """,
    }

    response = requests.post(PERTURBSEQR_URL, json=query)
    if not response.ok:
        raise APIFailure
    res = response.json()
    _check_gql_errors(res)

    consensus = res["data"]["currentBackground"]["pairedEnrich"]["consensus"]
    enrichment = res["data"]["currentBackground"]["pairedEnrich"]["nodes"]

    df_consensus_pair = pd.DataFrame(consensus).rename(
        columns={
            "drug": "perturbation",
            "pvalueUp": "pvalueMimic",
            "pvalueDown": "pvalueReverse",
            "adjPvalueUp": "adjPvalueMimic",
            "adjPvalueDown": "adjPvalueReverse",
            "oddsRatioUp": "oddsRatioMimic",
            "oddsRatioDown": "oddsRatioReverse",
        }
    )
    df_enrichment_pair = pd.DataFrame(enrichment)
    if df_enrichment_pair.empty:
        return df_enrichment_pair, df_consensus_pair

    df_enrichment_pair["term"] = df_enrichment_pair["geneSet"].map(lambda t: parse_perturbation_term(t["nodes"][0]["term"])[0])
    df_enrichment_pair["approved"] = df_enrichment_pair["geneSet"].map(
        lambda t: t["nodes"][0]["geneSetFdaCountsById"]["nodes"][0]["approved"]
    )
    df_enrichment_pair["count"] = df_enrichment_pair["geneSet"].map(
        lambda t: t["nodes"][0]["geneSetFdaCountsById"]["nodes"][0]["count"]
    )
    df_enrichment_pair["library"] = df_enrichment_pair["geneSet"].map(
        lambda t: t["nodes"][0].get("library", {}).get("name") if t["nodes"][0].get("library") else None
    )
    df_enrichment_pair = df_enrichment_pair.rename(columns={"term": "perturbation"}).drop(columns=["geneSet"])

    return df_enrichment_pair, df_consensus_pair


def get_perturbseqr_valid_genes(genes):
    """Given a list of candidate names, return the subset that are recognized
    as real gene symbols by Perturb-Seqr's own gene database. Used here to
    classify perturbation hits as 'gene KO' vs 'drug'."""
    if len(genes) == 0:
        return []
    query = {
        "query": """query GenesQuery($genes: [String]!) {
            geneMap2(genes: $genes) {
                nodes {
                    gene
                    geneInfo {
                        symbol
                        }
                    }
                }
            }""",
        "variables": {"genes": genes},
        "operationName": "GenesQuery",
    }
    response = requests.post(PERTURBSEQR_URL, json=query)
    if not response.ok:
        raise APIFailure
    res = response.json()
    _check_gql_errors(res)
    return [g["geneInfo"]["symbol"] for g in res["data"]["geneMap2"]["nodes"] if g["geneInfo"] is not None]


# Perturb-Seqr's dataset libraries, grouped by perturbation type. Matching is done
# by keyword (case-insensitive substring) rather than exact name, since the exact
# string returned by the API's `library.name` / `libraries` fields may be a shortened
# or differently-cased version of these display names.
GENE_LIBRARY_KEYWORDS = [
    "perturb atlas", "gene knockout", "l1000 gene", "l1000 xpr", "replogle",
    "creeds gene", "cm4ai", "rummageo gene",
]
DRUG_LIBRARY_KEYWORDS = [
    "l1000 chemical", "l1000 cp", "tahoe", "drug-seq", "moabox", "cmap",
    "ginkgo", "sciplex", "deepcover", "creeds drug", "creeds chem",
    "rummageo drug", "rummageo chem",
]

def classify_library_name(name):
    """Classify a single library name as gene (True), drug (False), or
    unrecognized (None) via keyword matching."""
    if not name:
        return None
    n = str(name).lower()
    is_gene_kw = any(k in n for k in GENE_LIBRARY_KEYWORDS)
    is_drug_kw = any(k in n for k in DRUG_LIBRARY_KEYWORDS)
    if is_gene_kw and not is_drug_kw:
        return True
    if is_drug_kw and not is_gene_kw:
        return False
    return None  # ambiguous or unrecognized -- caller should fall back

def classify_row_by_library(row):
    """Classify a row as gene (True)/drug (False)/unresolved (None) using
    whichever library column is present: a single 'library' string (enrichment,
    one row per signature) or a 'libraries' list (consensus, aggregated across
    all of a perturbation's signatures)."""
    lib = row.get("library")
    if isinstance(lib, str) and lib:
        return classify_library_name(lib)
    libs = row.get("libraries")
    if isinstance(libs, list) and len(libs) > 0:
        results = {classify_library_name(l) for l in libs}
        results.discard(None)
        if results == {True}:
            return True
        if results == {False}:
            return False
    return None


In [ ]:
# Run Perturb-Seqr for the mode selected above, and classify hits into drugs vs. KO genes.
#
# Consensus includes BOTH drug and gene-knockout perturbations under the same "drug"
# field name (per Perturb-Seqr's own docs) -- it is not drug-only. To make sure
# lower-ranked gene hits aren't missed, we pull consensus both unfiltered and with
# filterKo=True, then merge the unique candidates before classifying them below.
#
# Every hit is classified using two layers: first, the dataset library it came from
# (ground truth from Perturb-Seqr itself); second, for any hit whose library name
# doesn't clearly match, a fallback check against Perturb-Seqr's own gene database.

df_ranked = pd.DataFrame()
top_drugs, top_ko_genes = [], []
pvalue_col = odds_col = None
perturbseqr_ok = False

try:
    if mode == "single":
        df_enrichment, df_consensus_a = enrich_perturbseqr_single_set(input_genes, first=2000, filter_ko=False)
        _, df_consensus_b = enrich_perturbseqr_single_set(input_genes, first=2000, filter_ko=True)
    else:
        df_enrichment, df_consensus_a = enrich_perturbseqr_up_down(genes_up, genes_down, first=500, top_n=5000, filter_ko=False)
        _, df_consensus_b = enrich_perturbseqr_up_down(genes_up, genes_down, first=500, top_n=5000, filter_ko=True)

    df_consensus = pd.concat([df_consensus_a, df_consensus_b], ignore_index=True).drop_duplicates(subset=["perturbation"])

    if rank_by == "consensus":
        pvalue_col = "adjPvalueMimic" if "adjPvalueMimic" in df_consensus.columns else "adjPvalue"
        odds_col = "oddsRatioMimic" if "oddsRatioMimic" in df_consensus.columns else "oddsRatio"
        df_ranked = df_consensus.dropna(subset=["perturbation"]).copy() if "perturbation" in df_consensus.columns else df_consensus.copy()
    else:
        pvalue_col = "adjPvalueMimic" if "adjPvalueMimic" in df_enrichment.columns else "adjPvalue"
        odds_col = "oddsRatioMimic" if "oddsRatioMimic" in df_enrichment.columns else "oddsRatio"
        df_ranked = df_enrichment.dropna(subset=["perturbation"]).copy() if "perturbation" in df_enrichment.columns else df_enrichment.copy()

    if df_ranked.empty:
        raise NoResults

    # Sorted by significance, but NOT deduplicated -- every individual Perturb-Seqr
    # result is preserved, including repeats of the same drug/gene from different
    # cell lines or datasets.
    if pvalue_col in df_ranked.columns:
        df_ranked = df_ranked.sort_values(pvalue_col, ascending=True).reset_index(drop=True)

    all_candidates = df_ranked["perturbation"].tolist()
    # Deduplicate only for the gene-symbol lookup itself -- looking up "TP53" once
    # instead of once per cell line is purely a lookup-efficiency choice and doesn't
    # affect which rows end up in df_ranked or the top lists below.
    unique_candidates = list(dict.fromkeys(all_candidates))

    # --- Layer 1: classify by source library (ground truth from Perturb-Seqr) ---
    df_ranked["is_gene"] = df_ranked.apply(classify_row_by_library, axis=1)
    unresolved_mask = df_ranked["is_gene"].isna()
    unresolved_candidates = list(dict.fromkeys(df_ranked.loc[unresolved_mask, "perturbation"].tolist()))

    # --- Layer 2: fallback for anything the library couldn't confidently classify --
    # (missing library info, or a library name that didn't match a known keyword) --
    # check whether the name resolves to a real gene symbol via Perturb-Seqr's gene database.
    if unresolved_candidates:
        valid_genes_raw = get_perturbseqr_valid_genes(unresolved_candidates)
        valid_gene_symbols = {g.upper() for g in valid_genes_raw}

        # A few Connectivity Map datasets label knockout/knockdown perturbations with a
        # suffix (e.g. "TP53-KO", "shTP53"). If the raw name doesn't resolve, try a couple
        # of common normalized variants before giving up on it.
        def normalize_variants(name):
            variants = {name}
            stripped = re.sub(r"(?i)[\s_-]*(ko|kd|crispr|sg\d*|sh)$", "", name).strip()
            variants.add(stripped)
            variants.add(re.sub(r"(?i)^sh", "", name).strip())
            variants.add(re.sub(r"(?i)^sg\d*[\s_-]*", "", name).strip())
            return {v for v in variants if v}

        still_unmatched = [c for c in unresolved_candidates if c.upper() not in valid_gene_symbols]
        variant_lookup = {}
        for c in still_unmatched:
            for v in normalize_variants(c):
                if v != c:
                    variant_lookup.setdefault(v, []).append(c)

        if variant_lookup:
            extra_valid = get_perturbseqr_valid_genes(list(variant_lookup.keys()))
            for symbol in extra_valid:
                for original in variant_lookup.get(symbol, []) + variant_lookup.get(symbol.upper(), []):
                    valid_gene_symbols.add(original.upper())

        df_ranked.loc[unresolved_mask, "is_gene"] = df_ranked.loc[unresolved_mask, "perturbation"].map(
            lambda p: p.upper() in valid_gene_symbols
        )

    # Anything still unresolved (shouldn't normally happen) defaults to "drug" rather
    # than silently dropping the hit.
    df_ranked["is_gene"] = df_ranked["is_gene"].fillna(False).astype(bool)

    top_drugs = df_ranked.loc[~df_ranked["is_gene"], "perturbation"].tolist()[:top_n_drugs]
    top_ko_genes = df_ranked.loc[df_ranked["is_gene"], "perturbation"].tolist()[:top_n_ko_genes]
    perturbseqr_ok = True

    n_rows = len(df_ranked)
    n_unique = len(unique_candidates)
    n_unique_from_library = n_unique - len(unresolved_candidates)
    n_gene_rows = int(df_ranked["is_gene"].sum())
    n_drug_rows = n_rows - n_gene_rows
    print(f"Ranking by: {rank_by} ({pvalue_col})")
    print(f"Perturb-Seqr returned {n_rows} result(s) across {n_unique} unique drug(s)/gene(s) "
          f"-- repeats (e.g. the same gene hit from different cell lines/datasets) are kept, not collapsed.")
    print(f"Classified {n_unique_from_library}/{n_unique} unique names directly from their source library; "
          f"cross-checked the remaining {len(unresolved_candidates)} against Perturb-Seqr\'s gene database.")
    print(f"{n_gene_rows} result row(s) are KO genes, {n_drug_rows} are drugs.")

except APIFailure:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>Unable to retrieve results because of a bad response from the Perturb-Seqr API</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Please try again later.</div>"))
except NoResults:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>No perturbations were returned for this gene set</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Please try again with a different input list.</div>"))


In [ ]:
def build_perturbation_table(df, is_gene_flag, top_n, pvalue_col, odds_col):
    """Build a display-ready table for either the drug (is_gene_flag=False) or
    gene-KO (is_gene_flag=True) subset of df_ranked."""
    d = df[df["is_gene"] == is_gene_flag].head(top_n).copy()
    if d.empty:
        return d, pd.DataFrame()
    d["neg_log10_p"] = -np.log10(d[pvalue_col].astype(float).clip(lower=1e-300))
    table_dict = {
        "Rank": range(1, len(d) + 1),
        "Perturbation": d["perturbation"],
        "p-value": d[pvalue_col].map(lambda x: f"{x:.2e}"),
    }
    if odds_col in d.columns:
        table_dict["Odds Ratio"] = d[odds_col].round(2)
    lib_col = "library" if "library" in d.columns else ("libraries" if "libraries" in d.columns else None)
    if lib_col:
        table_dict["Library"] = d[lib_col].map(lambda x: ", ".join(x[:2]) if isinstance(x, list) else x)
    table = pd.DataFrame(table_dict)
    return d, table


def show_perturbseqr_section(d, table, title, xlabel, color, file_stub, fig_caption):
    """Display a table + bar chart for one of the two (drug / gene) categories.
    Returns the saved chart filename, or None if there were no results."""
    if d.empty:
        display(HTML("<div style=\'font-size:1.05rem; padding:0.5rem 0;\'><b>No results in this category for the current gene set.</b></div>"))
        return None
    display(HTML(f"<strong>{title}</strong>"))
    display(HTML(table.to_html(index=False)))
    make_bar_chart(
        table["Perturbation"], d["neg_log10_p"],
        title=title, xlabel=xlabel, colors=color
    )
    bar_file = f"{file_stub}.png".replace(" ", "_")
    plt.savefig(bar_file, bbox_inches="tight")
    plt.show()
    display(Markdown(fig_caption))
    return bar_file


drugs_df = genes_df = pd.DataFrame()
drugs_table = genes_table = pd.DataFrame()
drugs_bar_file = genes_bar_file = None

if perturbseqr_ok:
    drugs_df, drugs_table = build_perturbation_table(df_ranked, False, top_n_drugs, pvalue_col, odds_col)
    caption_drugs = (
        f"**Table 1a. Top {len(drugs_df)} drug perturbations for `{query_name}`,** "
        f"ranked by {pvalue_col} ({rank_by})."
    )
    drugs_bar_file = show_perturbseqr_section(
        drugs_df, drugs_table,
        title=f"Top {top_n_drugs} Drugs (Perturb-Seqr)",
        xlabel="-log10(p-value)", color="#4C72B0",
        file_stub=f"{query_name}_top_drugs_bar",
        fig_caption=f"**Figure 1a.** Top {len(drugs_df)} drug perturbations for `{query_name}`, ranked by -log10(p-value)."
    )
    display(Markdown(caption_drugs))

    genes_df, genes_table = build_perturbation_table(df_ranked, True, top_n_ko_genes, pvalue_col, odds_col)
    caption_genes = (
        f"**Table 1b. Top {len(genes_df)} gene knockout perturbations for `{query_name}`,** "
        f"ranked by {pvalue_col} ({rank_by})."
    )
    genes_bar_file = show_perturbseqr_section(
        genes_df, genes_table,
        title=f"Top {top_n_ko_genes} KO Genes (Perturb-Seqr)",
        xlabel="-log10(p-value)", color="#DD8452",
        file_stub=f"{query_name}_top_genes_bar",
        fig_caption=f"**Figure 1b.** Top {len(genes_df)} gene knockout perturbations for `{query_name}`, ranked by -log10(p-value)."
    )
    display(Markdown(caption_genes))


## Link to Perturb-Seqr

In [ ]:
perturbseqr_link = "https://perturbseqr.maayanlab.cloud"
if perturbseqr_ok:
    display(HTML(f"<div style=\'font-size:1.25rem; padding:1rem 0;\'><a href=\'{perturbseqr_link}\' target=\'_blank\'>Explore the complete Perturb-Seqr connectivity mapping database online.</a></div>"))
else:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>No Perturb-Seqr results available for the current query</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Please try again with a different input list.</div>"))


## Step 2: DrugEnrichr — Enrichment Analysis on Top Drugs

Runs the top drug hits from Perturb-Seqr (Step 1) through DrugEnrichr to find enriched mechanisms of action,
targets, and other drug-set annotations.

In [ ]:
DRUGENRICHR_ADD_URL = "https://maayanlab.cloud/DrugEnrichr/addList"
DRUGENRICHR_ENRICH_URL = "https://maayanlab.cloud/DrugEnrichr/enrich"
DRUGENRICHR_STATS_URL = "https://maayanlab.cloud/DrugEnrichr/datasetStatistics"

def drugenrichr_add_list(drugs, description="Top Perturb-Seqr drugs"):
    # DrugEnrichr\'s own example drug lists are all lowercase (e.g. "adoprazine",
    # "alisporivir"), suggesting it does exact/case-sensitive matching against its
    # internal drug identifiers. Perturb-Seqr drug names keep their original mixed
    # case (e.g. "Idarubicin", "(S)-(+)-Camptothecin"), which would silently match
    # nothing if submitted as-is -- so lowercase them here before submitting.
    drugs_str = "\n".join(d.lower() for d in drugs)
    payload = {
        "list": (None, drugs_str),
        "description": (None, description),
    }
    response = requests.post(DRUGENRICHR_ADD_URL, files=payload)
    if not response.ok:
        raise APIFailure
    return response.json()

def get_drugenrichr_libraries():
    """Fetch the list of valid backgroundType library names directly from
    DrugEnrichr, so we validate against ground truth instead of trusting a
    hardcoded name that may be stale. Returns None if the endpoint can\'t be
    reached, so callers can fall back gracefully rather than hard-failing."""
    try:
        response = requests.get(DRUGENRICHR_STATS_URL)
        response.raise_for_status()
        data = response.json()
        stats = data.get("statistics", data if isinstance(data, list) else [])
        libs = [s.get("libraryName") for s in stats if isinstance(s, dict) and s.get("libraryName")]
        return libs if libs else None
    except Exception as e:
        print(f"Could not fetch DrugEnrichr\'s library list ({e}); skipping validation.")
        return None

def drugenrichr_enrich(user_list_id, library="SIDER_Side_Effects"):
    # Validate the requested library against DrugEnrichr\'s live list before
    # querying -- an invalid/stale library name otherwise fails SILENTLY (the
    # API just omits that key from its response, and naive code defaults to an
    # empty list with no error at all).
    valid_libraries = get_drugenrichr_libraries()
    if valid_libraries is not None and library not in valid_libraries:
        close = difflib.get_close_matches(library, valid_libraries, n=3, cutoff=0.4)
        suggestion = f" Closest matches: {close}" if close else ""
        raise ValueError(
            f"\'{library}\' is not a recognized DrugEnrichr library.{suggestion} "
            f"See the full list via get_drugenrichr_libraries()."
        )

    query_string = f"?userListId={user_list_id}&backgroundType={library}"
    response = requests.get(DRUGENRICHR_ENRICH_URL + query_string)
    if not response.ok:
        raise APIFailure
    data = response.json()
    rows = data.get(library, [])
    if not rows:
        return pd.DataFrame()

    # DrugEnrichr's response schema (like Enrichr's) has been observed with either
    # 7 columns (no old-style p-value columns) or 9 (including old_pvalue /
    # old_adj_pvalue) -- detect from the actual row length rather than hardcoding
    # one, so a schema mismatch doesn't crash the whole cell.
    base_cols = ["rank", "term", "pvalue", "zscore", "combined_score", "overlapping_drugs", "adj_pvalue"]
    extended_cols = base_cols + ["old_pvalue", "old_adj_pvalue"]
    n = len(rows[0])
    if n == len(base_cols):
        cols = base_cols
    elif n == len(extended_cols):
        cols = extended_cols
    else:
        cols = [f"col_{i}" for i in range(n)]
        print(f"Warning: unexpected DrugEnrichr row length ({n}); using generic column names {cols}.")

    return pd.DataFrame(rows, columns=cols)


def build_enrichr_style_table(df, top_n, overlap_col, label_col="term"):
    """Shared display-table builder for DrugEnrichr/Enrichr results."""
    d = df.head(top_n).copy()
    if d.empty:
        return d, pd.DataFrame()
    d["neg_log10_p"] = -np.log10(d["pvalue"].astype(float).clip(lower=1e-300))
    table = pd.DataFrame({
        "Rank": range(1, len(d) + 1),
        "Term": d[label_col],
        "p-value": d["pvalue"].map(lambda x: f"{x:.2e}"),
        "Combined Score": d["combined_score"].round(2),
        "Overlap": d[overlap_col].map(lambda x: ", ".join(x[:6]) if isinstance(x, list) else str(x)),
    })
    return d, table


def show_enrichr_style_section(d, table, title, xlabel, color, file_stub, fig_caption):
    if d.empty:
        display(HTML("<div style=\'font-size:1.05rem; padding:0.5rem 0;\'><b>No enrichment terms were returned for this input set/library.</b></div>"))
        return None
    display(HTML(f"<strong>{title}</strong>"))
    display(HTML(table.to_html(index=False)))
    make_bar_chart(table["Term"], d["neg_log10_p"], title=title, xlabel=xlabel, colors=color)
    bar_file = f"{file_stub}.png".replace(" ", "_")
    plt.savefig(bar_file, bbox_inches="tight")
    plt.show()
    display(Markdown(fig_caption))
    return bar_file


In [ ]:
df_drugenrichr = pd.DataFrame()
drugenrichr_table = pd.DataFrame()
drugenrichr_bar_file = None
drugenrichr_ok = False

try:
    if len(top_drugs) == 0:
        raise NoResults

    drug_list_result = drugenrichr_add_list(top_drugs)
    df_drugenrichr = drugenrichr_enrich(drug_list_result["userListId"], library=drugenrichr_library)
    if df_drugenrichr.empty:
        raise NoResults
    df_drugenrichr = df_drugenrichr.sort_values("pvalue").reset_index(drop=True)
    drugenrichr_ok = True

    drugenrichr_d, drugenrichr_table = build_enrichr_style_table(df_drugenrichr, 15, "overlapping_drugs")
    caption_drugenrichr = (
        f"**Table 2. Top DrugEnrichr terms for the top drugs from `{query_name}`,** "
        f"enriched against the `{drugenrichr_library}` library."
    )
    drugenrichr_bar_file = show_enrichr_style_section(
        drugenrichr_d, drugenrichr_table,
        title=f"DrugEnrichr: {drugenrichr_library}",
        xlabel="-log10(p-value)", color="#55A868",
        file_stub=f"{query_name}_drugenrichr_bar",
        fig_caption=f"**Figure 2.** Top DrugEnrichr terms for `{query_name}`, ranked by -log10(p-value)."
    )
    display(Markdown(caption_drugenrichr))

except APIFailure:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>Unable to retrieve results because of a bad response from the DrugEnrichr API</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Please try again later.</div>"))
except NoResults:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>No DrugEnrichr results for the current top drugs / library</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Try a different library, or rerun Step 1 with different parameters.</div>"))


## Link to DrugEnrichr

In [ ]:
drugenrichr_link = "https://maayanlab.cloud/DrugEnrichr/"
if drugenrichr_ok:
    display(HTML(f"<div style=\'font-size:1.25rem; padding:1rem 0;\'><a href=\'{drugenrichr_link}\' target=\'_blank\'>Explore the complete DrugEnrichr database online.</a></div>"))
else:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>No DrugEnrichr results available for the current query</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Please try again with a different input list or library.</div>"))


## Step 3: Enrichr — Enrichment Analysis on Top KO Genes

Runs the top KO gene hits from Perturb-Seqr (Step 1) through Enrichr to find enriched pathways, ontologies, and
other gene-set annotations.

In [ ]:
ENRICHR_ADD_URL = "https://maayanlab.cloud/Enrichr/addList"
ENRICHR_ENRICH_URL = "https://maayanlab.cloud/Enrichr/enrich"

def enrichr_add_list(genes, description="Top Perturb-Seqr KO genes"):
    genes_str = "\n".join(genes)
    payload = {
        "list": (None, genes_str),
        "description": (None, description),
    }
    response = requests.post(ENRICHR_ADD_URL, files=payload)
    if not response.ok:
        raise APIFailure
    return response.json()

def enrichr_enrich(user_list_id, library="KEGG_2021_Human"):
    query_string = f"?userListId={user_list_id}&backgroundType={library}"
    response = requests.get(ENRICHR_ENRICH_URL + query_string)
    if not response.ok:
        raise APIFailure
    data = response.json()
    cols = ["rank", "term", "pvalue", "zscore", "combined_score", "overlapping_genes", "adj_pvalue", "old_pvalue", "old_adj_pvalue"]
    rows = data.get(library, [])
    return pd.DataFrame(rows, columns=cols)


In [ ]:
df_enrichr = pd.DataFrame()
enrichr_table = pd.DataFrame()
enrichr_bar_file = None
enrichr_ok = False

try:
    if len(top_ko_genes) == 0:
        raise NoResults

    gene_list_result = enrichr_add_list(top_ko_genes)
    df_enrichr = enrichr_enrich(gene_list_result["userListId"], library=enrichr_library)
    if df_enrichr.empty:
        raise NoResults
    df_enrichr = df_enrichr.sort_values("pvalue").reset_index(drop=True)
    enrichr_ok = True

    enrichr_d, enrichr_table = build_enrichr_style_table(df_enrichr, 15, "overlapping_genes")
    caption_enrichr = (
        f"**Table 3. Top Enrichr terms for the top KO genes from `{query_name}`,** "
        f"enriched against the `{enrichr_library}` library."
    )
    enrichr_bar_file = show_enrichr_style_section(
        enrichr_d, enrichr_table,
        title=f"Enrichr: {enrichr_library}",
        xlabel="-log10(p-value)", color="#C44E52",
        file_stub=f"{query_name}_enrichr_bar",
        fig_caption=f"**Figure 3.** Top Enrichr terms for `{query_name}`, ranked by -log10(p-value)."
    )
    display(Markdown(caption_enrichr))

except APIFailure:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>Unable to retrieve results because of a bad response from the Enrichr API</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Please try again later.</div>"))
except NoResults:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>No Enrichr results for the current top KO genes / library</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Try a different library, or rerun Step 1 with different parameters.</div>"))


## Link to Enrichr

In [ ]:
enrichr_link = "https://maayanlab.cloud/Enrichr/"
if enrichr_ok:
    display(HTML(f"<div style=\'font-size:1.25rem; padding:1rem 0;\'><a href=\'{enrichr_link}\' target=\'_blank\'>Explore the complete Enrichr database online.</a></div>"))
else:
    display(HTML("<div style=\'font-size:1.5rem; padding:1rem 0;\'><b>No Enrichr results available for the current query</b></div>"))
    display(HTML("<div style=\'font-size:1rem; padding:1rem 0;\'>Please try again with a different input list or library.</div>"))


## Save Full Pipeline Report

Combine the results and charts from all three steps (Perturb-Seqr, DrugEnrichr, Enrichr) into a single standalone
HTML report file that can be downloaded and shared.

In [ ]:
def _safe_table_html(table, caption, image_file=None):
    if table is not None and len(table) > 0:
        img_html = f"<img src=\'{image_file}\' style=\'max-width:100%; margin-top:1rem;\'>" if image_file else ""
        return f"<div>{table.to_html(index=False)}</div>{img_html}<p>{caption}</p>"
    else:
        return "<p><i>No results were available for this step.</i></p>"

report_sections = []
report_sections.append("<h2>Step 1: Perturb-Seqr &mdash; Drug &amp; Gene Perturbation Discovery</h2>")
report_sections.append("<h3>Top Drugs</h3>")
report_sections.append(_safe_table_html(drugs_table, caption_drugs if perturbseqr_ok else "", image_file=drugs_bar_file))
report_sections.append("<h3>Top KO Genes</h3>")
report_sections.append(_safe_table_html(genes_table, caption_genes if perturbseqr_ok else "", image_file=genes_bar_file))

report_sections.append("<h2>Step 2: DrugEnrichr &mdash; Enrichment Analysis on Top Drugs</h2>")
report_sections.append(_safe_table_html(drugenrichr_table, caption_drugenrichr if drugenrichr_ok else "", image_file=drugenrichr_bar_file))

report_sections.append("<h2>Step 3: Enrichr &mdash; Enrichment Analysis on Top KO Genes</h2>")
report_sections.append(_safe_table_html(enrichr_table, caption_enrichr if enrichr_ok else "", image_file=enrichr_bar_file))

link_items = []
if perturbseqr_ok:
    link_items.append(f"<li><a href=\'{perturbseqr_link}\' target=\'_blank\'>Perturb-Seqr</a></li>")
if drugenrichr_ok:
    link_items.append(f"<li><a href=\'{drugenrichr_link}\' target=\'_blank\'>DrugEnrichr</a></li>")
if enrichr_ok:
    link_items.append(f"<li><a href=\'{enrichr_link}\' target=\'_blank\'>Enrichr</a></li>")
if link_items:
    link_items_html = "".join(link_items)
    report_sections.append(f"<h2>Explore Further</h2><ul>{link_items_html}</ul>")

report_sections_html = "".join(report_sections)
report_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

html_report = f"""<html>
<head>
<meta charset="utf-8">
<title>Drug & Gene Perturbation Discovery Report - {query_name}</title>
<style>
  body {{ font-family: Arial, sans-serif; margin: 2rem; color: #222; }}
  table {{ border-collapse: collapse; margin-bottom: 1rem; }}
  th, td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
  th {{ background-color: #f2f2f2; }}
  h1 {{ border-bottom: 2px solid #333; padding-bottom: 0.5rem; }}
  h2 {{ margin-top: 2rem; color: #333; }}
  h3 {{ margin-top: 1.5rem; color: #555; }}
</style>
</head>
<body>
<h1>Drug & Gene Perturbation Discovery Report: {query_name}</h1>
<p>Generated on {report_timestamp}</p>
{report_sections_html}
</body>
</html>
"""

report_filename = f"{query_name}_pipeline_report.html".replace(" ", "_")
with open(report_filename, "w") as f:
    f.write(html_report)

display(HTML(f"<div style=\'font-size:1.25rem; padding:1rem 0;\'>Full pipeline report saved to <code>{report_filename}</code></div>"))
display(HTML(f"<div>Download full report: <a href=\'{report_filename}\' target=_blank>{report_filename}</a></div>"))


---
### Citations
- Gardner JK, Taub LD, Clarke DJB, Diamant I, Ma'ayan A. Perturb-Seqr: Comprehensive Signature Search Engine Integrating Connectivity Maps from Multiple Sources. https://perturbseqr.maayanlab.cloud/
- Kuleshov MV, et al. modEnrichr: a suite of gene set enrichment analysis tools for model organisms. Nucleic Acids Res. 2019.
- Chen EY, et al. Enrichr: interactive and collaborative HTML5 gene list enrichment analysis tool. BMC Bioinformatics. 2013;128(14)
- Kuleshov MV, et al. Enrichr: a comprehensive gene set enrichment analysis web server 2016 update. Nucleic Acids Research. 2016; gkw377.
